In [1]:
! pip install predictionguard langchain

In [2]:
import os
import json

from predictionguard import PredictionGuard
from langchain import PromptTemplate
from langchain import PromptTemplate, FewShotPromptTemplate
import numpy as np
from getpass import getpass

In [4]:
pg_access_token = getpass('Enter your Prediction Guard access api key: ')
os.environ['PREDICTIONGUARD_API_KEY'] = pg_access_token

In [5]:
client = PredictionGuard()

## Zero Shot

In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

def evaluate_predictions(actual_df, prediction_df):
    """
    Evaluate the performance of predicted labels against actual labels.

    Args:
        actual_df (pd.DataFrame): DataFrame containing the actual labels with "ReviewText" and "label".
        prediction_df (pd.DataFrame): DataFrame containing the predicted labels with "ReviewText" and "PredictedLabel".

    Returns:
        dict: A dictionary containing accuracy, precision, recall, and F1 score.
    """
    # Combine the actual and predicted DataFrames on "ReviewText"
    comparison_df = pd.merge(
        actual_df, 
        prediction_df, 
        on="ReviewText", 
        how="inner"
    )

    if comparison_df.empty:
        raise ValueError("No matching ReviewText found between actual and predicted data.")

    # Check for matches between 'label' and 'PredictedLabel'
    comparison_df["Match"] = comparison_df["label"] == comparison_df["PredictedLabel"]

    # Calculate accuracy
    accuracy = comparison_df["Match"].mean()

    # Convert labels to numeric for precision, recall, and F1 score
    y_true = comparison_df["label"]
    y_pred = comparison_df["PredictedLabel"]

    # Calculate evaluation metrics
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    # Print classification report for more insights (optional)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    # Return the evaluation metrics
    return {
        "accuracy": accuracy * 100,
        "precision": precision * 100,
        "recall": recall * 100,
        "f1_score": f1
    }

In [32]:
import time
import pandas as pd
from langchain.prompts import FewShotPromptTemplate, PromptTemplate

# Define a demo formatter template for product reviews.
demo_formatter_template = """\nReview: {review}
Categories: {categories}
Class: {class}\n"""

# Define a prompt template for the demonstrations.
demo_prompt = PromptTemplate(
    input_variables=["review", "categories", "class"],
    template=demo_formatter_template,
)

# Each row here includes:
# 1. Example review text
# 2. Categories for the review
# 3. The expected class label
few_examples = [
    ["I put this into a 1999 Mazda Miata MX-5. It fit perfectly. It came fully charged. Delivered it saved me $50. Great deal. Love it!", 
     "Correct Size, Wrong Size, No Comment", "Correct Size"],
    ["I ordered this product for my car. Amazon says this will fit my 1996 Nissan Sentra GXE auto transmission. When the part arrived, I knew it wouldn't fit. I won't order parts online again.", 
     "Correct Size, Wrong Size, No Comment", "Wrong Size"],
    ["The battery was packaged well for shipping. It was as described, but I haven't installed it yet.", 
     "Correct Size, Wrong Size, No Comment", "No Comment"]
]

# Convert few_examples into the required format
examples = []
for ex in few_examples:
    examples.append({
        "review": ex[0],
        "categories": ex[1],
        "class": ex[2]
    })

# Define the FewShotPromptTemplate
few_shot_prompt = FewShotPromptTemplate(

    # This is the demonstration data we want to insert into the prompt.
    examples=examples,
    example_prompt=demo_prompt,
    example_separator="\n",

    # This is the boilerplate portion of the prompt corresponding to
    # the prompt task instructions.
    prefix="Classify the following product reviews into one of the given categories. Only output one of the provided categories for the class corresponding to each review.\n",

    # The suffix of the prompt is where we will put the output indicator
    # and define where the "on-the-fly" user input would go.
    suffix="\nReview: {review}\nCategories: {categories}\nClass: ",
    input_variables=["review", "categories"],
)

# Load the first 100 reviews from fit.csv
fit_df = pd.read_csv("fit.csv").head(500)

# Create a list to store predictions
predictions = []

# Iterate through the first 100 reviews
for _, row in fit_df.iterrows():
    review = row["ReviewText"]
    categories = "Correct Size, Wrong Size, No Comment"
    
    # Generate the prompt
    myprompt = few_shot_prompt.format(review=review, categories=categories)
    
    # Send the prompt to the model
    response = client.chat.completions.create(
        model="Hermes-3-Llama-3.1-70B",
        messages=[{"role": "user", "content": myprompt}]
    )
    
    # Extract the model's prediction
    predicted_label = response['choices'][0]['message']['content'].strip()
    predictions.append({"ReviewText": review, "PredictedLabel": predicted_label})
    print(predicted_label)
    
    time.sleep(2)

# Convert predictions to a DataFrame and save to CSV
predictions_df = pd.DataFrame(predictions)
predictions_df.to_csv("predictionguard_predictions_first_100_reviews.csv", index=False)

print("Predictions saved to predictionguard_predictions_first_100_reviews.csv")

No Comment
No Comment
Correct Size
Correct Size
No Comment
No Comment
Correct Size
Correct Size
No Comment
No Comment
No Comment
No Comment
No Comment
Wrong Size
Correct Size
No Comment
No Comment
No Comment
Correct Size
Correct Size
Correct Size
No Comment
Correct Size
Correct Size
No Comment
Correct Size
No Comment
Correct Size
No Comment
Correct Size
No Comment
No Comment
Correct Size
Correct Size
No Comment
No Comment
Correct Size
Correct Size
No Comment
Correct Size
No Comment
No Comment
No Comment
No Comment
Correct Size
No Comment
No Comment
No Comment
No Comment
Correct Size
Correct Size
Wrong Size
No Comment
Correct Size
Correct Size
Correct Size
Correct Size
No Comment
No Comment
No Comment
Correct Size
No Comment
Correct Size
No Comment
No Comment
Wrong Size
No Comment
Correct Size
No Comment
Wrong Size
Correct Size
No Comment
No Comment
Correct Size
No Comment
Wrong Size
No Comment
No Comment
No Comment
Wrong Size
No Comment
No Comment
No Comment
No Comment
Wrong Size
Wrong

In [33]:
input_file = "fit.csv"  # Original file with actual labels
predictions_file = "predictionguard_predictions_first_100_reviews.csv"  # File with model predictions

# Load input and prediction files as DataFrames
df_input = pd.read_csv(input_file).head(500)
df_predictions = pd.read_csv(predictions_file)

# Evaluate predictions
metrics = evaluate_predictions(df_input, df_predictions)

# Print evaluation metrics
print(f"Accuracy: {metrics['accuracy']:.2f}%")
print(f"Precision: {metrics['precision']:.2f}%")
print(f"Recall: {metrics['recall']:.2f}%")
print(f"F1 Score: {metrics['f1_score']:.2f}")


Classification Report:
              precision    recall  f1-score   support

Correct Size       0.47      0.75      0.58       134
  No Comment       0.83      0.58      0.68       317
  Wrong Size       0.59      0.71      0.65        49

    accuracy                           0.64       500
   macro avg       0.63      0.68      0.64       500
weighted avg       0.71      0.64      0.65       500

Accuracy: 64.20%
Precision: 62.82%
Recall: 68.39%
F1 Score: 0.64
